In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import cumulative_trapezoid
from scipy.signal import butter, filtfilt

ACCEL_COL = "Linear Acceleration y (m/s^2)"
TIME_COL = "Time (s)"

df = pd.read_excel("Data from phyphox.xlsx")

In [ ]:
start_time = 9.3
end_time = 15

#creates the conditions to cut the excess data
condition = (df[TIME_COL] >= start_time) & (df[TIME_COL] <= end_time)
extracted_df = df.loc[condition, [TIME_COL, ACCEL_COL]].copy() 

# Work on this dataframe directly from here on - no need to round-trip
# through CSV until we're done cleaning it.
df = extracted_df.dropna().sort_values(by=TIME_COL).reset_index(drop=True)


# Fix sensor spikes (outlier removal)
# Removes extreme anomalies (e.g., sensor drops or electrical pops)
z_scores = (df[ACCEL_COL] - df[ACCEL_COL].mean()) / df[ACCEL_COL].std()
is_outlier = np.abs(z_scores) > 3

rolling_median = df[ACCEL_COL].rolling(window=5, center=True).median()
df.loc[is_outlier, ACCEL_COL] = rolling_median[is_outlier]
df = df.dropna().reset_index(drop=True)  # clear any edges left by the rolling window


# Apply low-pass filtering
# Acceleration data is prone to picking up vibrations.
# A Butterworth filter keeps real motion and cuts out the vibrations
# ---------------------------------------------------------------------------
time_deltas = df[TIME_COL].diff().dropna()
sampling_freq = 1.0 / time_deltas.mean()  # e.g., if delta is 0.01s, freq is 100Hz

cutoff_freq = 5  # Hz - adjust based on your movement type
order = 2

def butter_lowpass_filter(data, cutoff, nyq_freq, order = 2):
    normal_cutoff = cutoff / nyq_freq
    b, a = butter(order, normal_cutoff, btype="low", analog=False)
    return filtfilt(b, a, data)  # filtfilt avoids time-shifting/phase lag


nyquist = 0.5 * sampling_freq

if cutoff_freq < nyquist:
    df["cleaned_acceleration"] = butter_lowpass_filter(
        df[ACCEL_COL].to_numpy(), cutoff_freq, nyquist, order
    )
else:
    df["cleaned_acceleration"] = df[ACCEL_COL].rolling(window=5, center=True).mean()
    df = df.dropna().reset_index(drop=True)

# Save the fully cleaned dataset, after all cleaning steps are done.
df.to_csv("Clean_Data.csv", index=False)

In [ ]:
#Read clean data and perform numerical integration to determine Velocity and Postion
df = pd.read_csv("Clean_Data.csv")

t = df[TIME_COL].to_numpy()
a_raw = df[ACCEL_COL].to_numpy()
a_clean = df["cleaned_acceleration"].to_numpy()

v = cumulative_trapezoid(a_clean, t, initial=0)
y = cumulative_trapezoid(v, t, initial=0)

df["Velocity (m/s)"] = v
df["Position (m)"] = y
df.to_csv("Skateboard Motion data.csv", index=False)

In [ ]:
plt.figure(figsize=(30, 24))
plt.subplots_adjust(hspace=0.5)
plt.grid(True, linewidth=0.8, alpha=0.6)
plt.axhline(0, color="black", linewidth=3)
plt.scatter(t, a_clean, s = 50, alpha = 0.8, color = "purple")
plt.xlabel("Time (s)", fontsize=40)
plt.ylabel("Acceleration (m/s^2)", fontsize=40)
plt.xticks(fontsize=25)
plt.yticks(fontsize=25)
plt.title("Acceleration of a skateboard vs Time", fontsize=60)
plt.subplots_adjust(hspace=0.5)
plt.tight_layout()

In [ ]:
plt.figure(figsize=(30, 24))
plt.grid(True, linewidth=0.8, alpha=0.6)
plt.axhline(0, color="black", linewidth=3)
plt.scatter(t, v, s = 40, alpha = 0.7, color = "green")
plt.xlabel("Time (s)", fontsize=40)
plt.ylabel("Velocity (m/s)", fontsize=40)
plt.xticks(fontsize=25)
plt.yticks(fontsize=25)
plt.title("Velocity of a skateboard vs Time", fontsize=60)
plt.subplots_adjust(hspace=0.5)

In [ ]:
plt.figure(figsize=(30, 24))
plt.grid(True, linewidth=0.8, alpha=0.6)
plt.axhline(0, color="black", linewidth=3)
plt.scatter(t, y, s = 5, alpha = 0.8, color = "red")
plt.xlabel("Time (s)", fontsize=40)
plt.ylabel("Position (m)", fontsize=40)
plt.xticks(fontsize=25)
plt.yticks(fontsize=25)
plt.title("Position of a skateboard vs Time", fontsize=60)
plt.subplots_adjust(hspace=0.5)